# Food Waste Reducer - YOLO training (Colab, one click)

> **GATE: do not run this until ML QA has re-verified ML-21 (near-duplicate leakage) on the session-grouped split.**
> Cell 4 enforces the mechanical part: it stops unless `dataset_report.json` says `split_grouping: "session"` AND `ml/eval/leakage_check.py` finds 0 held-out images within 300 s of a train/val image.

## What the owner clicks
1. **Runtime -> Change runtime type -> T4 GPU.**
2. Left sidebar **Secrets** (key icon): add `ROBOFLOW_API_KEY` and `KAGGLE_API_TOKEN`; switch **Notebook access** on for each. (`KAGGLE_API_TOKEN` is the Kaggle access token, used as a Bearer token; optional: without it the Kaggle source is skipped.)
3. **Runtime -> Run all.** Allow Drive access if asked (optional, only used to keep the run).
4. When it finishes, the last cell downloads **`export.zip`** (detector.pt, freshness.pt, class_names.json, predictions.jsonl, heldout_gt.jsonl, heldout_manifest.csv, dataset_report.json). Give it to the team.

Expect roughly 1-2 h on a T4 for the full run (80 epochs yolo26s + 30 epochs classifier). Ultralytics is **AGPL-3.0**; see `docs/ML.md`. Never paste keys into cells.

In [ ]:
# 1. Setup: code, deps, optional Drive
import os, subprocess, sys
REPO_URL = "https://github.com/<OWNER>/<REPO>.git"   # TODO owner: set this, or upload the ml/ folder to /content/repo/ml
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
%cd /content/repo/ml/yolo
!pip install -q -r requirements.txt roboflow kaggle
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"   # expect 8.4.175

SAVE_DIR = "/content/runs"          # becomes a Drive folder if Drive mounts
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/fwr_runs"
    os.makedirs(SAVE_DIR, exist_ok=True)
except Exception as e:               # Drive is optional
    print("Drive not mounted, runs stay in /content/runs:", e)
print("runs dir:", SAVE_DIR)

In [ ]:
# 2. Secrets (Colab Secrets only; values are never printed)
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
for k in ("ROBOFLOW_API_KEY", "KAGGLE_API_TOKEN"):
    v = secret(k)
    if v: os.environ[k] = v
    print(k, "set" if v else "MISSING")
assert os.environ.get("ROBOFLOW_API_KEY"), "Add ROBOFLOW_API_KEY in Colab Secrets (left sidebar) and re-run."
HAVE_KAGGLE = bool(os.environ.get("KAGGLE_API_TOKEN"))

In [ ]:
# 3. Download the four sources and write a Colab sources.json derived from the repo's one
import json, pathlib, zipfile, urllib.request
RAW = pathlib.Path("/content/raw"); RAW.mkdir(exist_ok=True)
srcs = {s["name"]: s for s in json.load(open("sources.json"))["sources"]}

# 3a. VegQual (figshare, CC BY 4.0)
vq = RAW / "vegqual"
if not vq.exists():
    vq.mkdir()
    urllib.request.urlretrieve("https://ndownloader.figshare.com/files/62799448", "/content/vegqual.zip")
    zipfile.ZipFile("/content/vegqual.zip").extractall(vq)
    os.remove("/content/vegqual.zip")
srcs["vegqual"]["root"] = str(next(vq.rglob("data.yaml")).parent)

# 3b. Roboflow (YOLO txt export). Versions are pinned in sources.json (fridge_objects v1, hien_fridge v12).
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
for name in ("fridge_objects", "hien_fridge"):
    cfg = srcs[name]["roboflow"]
    dest = RAW / name
    if not dest.exists():
        rf.workspace(cfg["workspace"]).project(cfg["project"]).version(cfg["version"]).download("yolov8", location=str(dest))
    srcs[name]["root"] = str(dest); srcs[name].pop("pending", None)
    got = [l for l in open(dest / "data.yaml") if l.startswith("names")]
    print(name, "v%s" % cfg["version"], got)      # must match the class_map keys in sources.json

# 3c. raghavrpotdar via the Kaggle REST API (Bearer token, CC0), only if the secret exists
k = srcs["fresh_stale_fruits_veg"]
if HAVE_KAGGLE:
    kd = RAW / "fresh_stale_fruits_veg"
    if not kd.exists():
        kd.mkdir()
        !curl -sL -H "Authorization: Bearer $KAGGLE_API_TOKEN" -o /content/kaggle.zip https://www.kaggle.com/api/v1/datasets/download/raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables
        zipfile.ZipFile("/content/kaggle.zip").extractall(kd)
        os.remove("/content/kaggle.zip")
    k["root"] = str(next(kd.rglob("fresh_apple")).parent); k.pop("pending", None)
    print("kaggle class dirs:", sorted(p.name for p in pathlib.Path(k["root"]).iterdir() if p.is_dir()))
else:
    k["pending"] = True
    print("KAGGLE_API_TOKEN missing: skipping fresh_stale_fruits_veg")

json.dump({"sources": list(srcs.values())}, open("/content/sources.colab.json", "w"), indent=1)

In [ ]:
# 4. Build the dataset and CHECK THE GATE (ML-21)
!python prepare_dataset.py --sources /content/sources.colab.json --out /content/fwr_data
rep = json.load(open("/content/fwr_data/dataset_report.json"))
assert rep.get("split_grouping") == "session", "Split is not session-grouped (ML-21). Stop: update the repo."
print("active classes:", rep["n_active_classes"], rep["active_classes"])
print("images:", {k: v["images"] for k, v in rep["splits"].items()})
print("dropped:", rep["dropped_images"])
print("cells not measurable (single session / <10 images):", rep["stratification"]["single_group_cells_kept_in_train_not_measurable"],
      rep["stratification"]["not_measurable_cells_lt_min_images"])
print("leakage by source:", json.dumps(rep["leakage_by_source"], indent=1))
!python /content/repo/ml/eval/leakage_check.py /content/fwr_data --json /content/leakage.json
lk = json.load(open("/content/leakage.json"))
assert lk["within_300s"] == 0, "Held-out images have train/val neighbours within 300 s: stop (ML-21)."

In [ ]:
# 5. Train (detector 80 ep yolo26s, classifier 30 ep) and export predictions
!python train.py --data /content/fwr_data --epochs 80 --cls-epochs 30 --device auto --project /content/repo/ml/yolo/runs

In [ ]:
# 6. Quick held-out check, keep the run in Drive (if mounted), zip the export
from ultralytics import YOLO
m = YOLO("runs/export/detector.pt")
print(m.val(data="/content/fwr_data/data.yaml", split="test"))   # held-out incl. synthetic low/warm copies; leaky-free only if QA re-verified ML-21
!cp -r runs/export {SAVE_DIR}/export_$(date +%Y%m%d_%H%M) || true
!cd runs && zip -r export.zip export
from google.colab import files
files.download("runs/export.zip")